In [ ]:
#------------------------------【Import packages】------------------------------
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
#------------------------------【adata info】------------------------------
adata_hvg = sc.read_h5ad("/staging/biology/jane0528/NMOSD/scRNA/Dataset/My_merged_protein_coding_genes/My_merge_PCA_2000HVG(protein_coding).h5ad")
print(adata_hvg)

# cell info
print("obs (cells × metadata)：", adata_hvg.obs.shape)
print(adata_hvg.obs.head())
print("----------------------------------------------------------------")
print(adata_hvg.obs.tail())

# pc info
print("\nvarm keys：", list(adata_hvg.varm.keys()))
print("PC loading：\n", adata_hvg.varm['PCs'][:5, :10]) # gene PCA


In [ ]:
#------------------------------【Data】------------------------------
#使用All PCs
pcs = adata_hvg.obsm["X_pca"] #All PCs
print(pcs.shape)

# 轉成DataFrame
pc_cols = [f"PC{i+1}" for i in range(pcs.shape[1])]
pca_data = pd.DataFrame(pcs, index=adata_hvg.obs_names, columns=pc_cols) #index ->cell 

# 加一個欄位做 NMOSD/Control 分組
pca_data["condition"] = adata_hvg.obs["Condition"].astype(str)

In [ ]:
#-----------------------------------【PCA圖 標記NMOSD/Control】-----------------------------------
fig, ax = plt.subplots(figsize=(6, 5))

sns.scatterplot(
    data=pca_data,
    x="PC1", y="PC2",
    hue="condition",
    ax=ax,
    s=15,
    palette={
        "Control":"#A1D4E8", 
        "NMOSD": "#EACDE4"
    },
    edgecolor=None,  # 移除邊框     
    alpha=0.8
)

# 標題與軸設定
ax.set_title("NMOSD/Control PCA", fontsize=14)
ax.set_xlabel("PC1", fontsize=10)
ax.set_ylabel("PC2", fontsize=10)
ax.tick_params(labelsize=9)

# 調整 legend 位置
legend = ax.legend(
    title="Condition",
    title_fontsize=9,
    fontsize=9,
    loc="upper left",
    bbox_to_anchor=(1.01, 1.01)
)

plt.tight_layout()
plt.show()


In [ ]:
#-----------------------------------【UMAP圖 標記NMOSD/Control】-----------------------------------

sc.pp.neighbors(adata_hvg, use_rep="X_pca")
sc.tl.umap(adata_hvg)
sc.pl.umap(
    adata_hvg,
    color="Condition",
    title=f"NMOSD/Control UMAP",
    legend_loc="right margin",
    legend_fontsize=8,
    frameon=True,#圖加框
    legend_fontoutline=1, #圖例設置為圓點+標籤
    palette={
        "Control":"#A1D4E8", 
        "NMOSD": "#EACDE4"
    }
)

In [ ]:
#-----------------------------------【PC9 標記NMOSD/Control】-----------------------------------
#建一個新的adata，避免覆蓋
adata_pc9 = adata_hvg.copy()
# 取前10個PC
pcs_9 = adata_pc9.obsm["X_pca"][:, :9]
print(pcs_9.shape)  # (細胞數, 9)

# 轉成DataFrame
pc_cols_9 = [f"PC{i+1}" for i in range(9)]
pca_data_9 = pd.DataFrame(pcs_9, index=adata_pc9.obs_names, columns=pc_cols_9)
pca_data_9["condition"] = adata_pc9.obs["Condition"].astype(str)


fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(
    data=pca_data_9,
    x="PC1", y="PC2",
    hue="condition",
    ax=ax,
    s=15,
    palette={
        "Control":"#A1D4E8", 
        "NMOSD": "#EACDE4"
    },
    edgecolor=None,
    alpha=0.8
)
ax.set_title("NMOSD/Control PCA (PC9)", fontsize=14)
ax.set_xlabel("PC1", fontsize=10)
ax.set_ylabel("PC2", fontsize=10)
ax.tick_params(labelsize=9)
ax.legend(title="Condition", title_fontsize=9, fontsize=9, loc="upper left", bbox_to_anchor=(1.01, 1.01))
plt.tight_layout()
plt.show()

In [ ]:
#-----------------------------------【UMAP圖 標記NMOSD/Control】-----------------------------------
# 使用前9個PC構建鄰近圖
sc.pp.neighbors(adata_pc9, n_pcs=9, use_rep="X_pca")

# 重跑 UMAP
sc.tl.umap(adata_pc9)

# 畫 UMAP
sc.pl.umap(
    adata_pc9,
    color="Condition",
    title="NMOSD/Control UMAP (PC9)",
    legend_loc="right margin",
    legend_fontsize=8,
    frameon=True,
    legend_fontoutline=1,
    palette={
        "Control":"#A1D4E8", 
        "NMOSD": "#EACDE4"
    }
)